# 02 — Bronze Layer (Auto Loader → VARIANT)

Uses Auto Loader (`cloudFiles`) in batch mode to read NDJSON files from the
Volume landing zone and write VARIANT-based bronze Delta tables.

Each bronze table has:
- `data VARIANT` — the full raw JSON object (semi-structured)
- `_ingestion_timestamp` — when the row was loaded
- `_source_file` — which file the row came from

Silver layer will extract typed columns using `data:field::type` syntax.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "nil_sponsorships")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

print(f"Bronze schema: {UC_CATALOG}.{BRONZE_SCHEMA}")
print(f"Volume path:   {VOLUME_PATH}")

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Bronze schema: alexander_booth.nil_sponsorships_bronze
Volume path:   /Volumes/alexander_booth/nil_sponsorships_bronze/raw_data


In [2]:
ENTITIES = ["athletes", "sponsors", "deals", "campaigns"]

for entity in ENTITIES:
    source_path     = f"{VOLUME_PATH}/{entity}/"
    checkpoint_path = f"{VOLUME_PATH}/_checkpoints/{entity}"
    target_table    = f"{UC_CATALOG}.{BRONZE_SCHEMA}.{entity}_raw"

    print(f"\nLoading {entity} from {source_path}")
    print(f"  Checkpoint: {checkpoint_path}")
    print(f"  Target:     {target_table}")

    query = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", checkpoint_path + "/_schema")
        .load(source_path)
        .selectExpr(
            "PARSE_JSON(TO_JSON(STRUCT(*))) AS data",
            "current_timestamp() AS _ingestion_timestamp",
            "_metadata.file_path AS _source_file"
        )
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(target_table)
    )

    query.awaitTermination()
    print(f"  Done: {entity}_raw")


Loading athletes from /Volumes/alexander_booth/nil_sponsorships_bronze/raw_data/athletes/
  Checkpoint: /Volumes/alexander_booth/nil_sponsorships_bronze/raw_data/_checkpoints/athletes
  Target:     alexander_booth.nil_sponsorships_bronze.athletes_raw


  Done: athletes_raw

Loading sponsors from /Volumes/alexander_booth/nil_sponsorships_bronze/raw_data/sponsors/
  Checkpoint: /Volumes/alexander_booth/nil_sponsorships_bronze/raw_data/_checkpoints/sponsors
  Target:     alexander_booth.nil_sponsorships_bronze.sponsors_raw


  Done: sponsors_raw

Loading deals from /Volumes/alexander_booth/nil_sponsorships_bronze/raw_data/deals/
  Checkpoint: /Volumes/alexander_booth/nil_sponsorships_bronze/raw_data/_checkpoints/deals
  Target:     alexander_booth.nil_sponsorships_bronze.deals_raw


  Done: deals_raw

Loading campaigns from /Volumes/alexander_booth/nil_sponsorships_bronze/raw_data/campaigns/
  Checkpoint: /Volumes/alexander_booth/nil_sponsorships_bronze/raw_data/_checkpoints/campaigns
  Target:     alexander_booth.nil_sponsorships_bronze.campaigns_raw


  Done: campaigns_raw


In [3]:
print("\n" + "=" * 60)
print("BRONZE LAYER SUMMARY")
print("=" * 60)

for entity in ENTITIES:
    table = f"{UC_CATALOG}.{BRONZE_SCHEMA}.{entity}_raw"
    count = spark.table(table).count()
    print(f"  {table}: {count:,} rows")

print("\nSample athletes_raw:")
spark.sql(f"""
    SELECT data:first_name::string AS first_name,
           data:last_name::string AS last_name,
           data:school::string AS school,
           data:sport::string AS sport,
           _ingestion_timestamp
    FROM {UC_CATALOG}.{BRONZE_SCHEMA}.athletes_raw
    LIMIT 5
""").show(truncate=False)


BRONZE LAYER SUMMARY


  alexander_booth.nil_sponsorships_bronze.athletes_raw: 200 rows


  alexander_booth.nil_sponsorships_bronze.sponsors_raw: 50 rows


  alexander_booth.nil_sponsorships_bronze.deals_raw: 300 rows


  alexander_booth.nil_sponsorships_bronze.campaigns_raw: 800 rows

Sample athletes_raw:


+----------+---------+----------------------+------------------+-----------------------+
|first_name|last_name|school                |sport             |_ingestion_timestamp   |
+----------+---------+----------------------+------------------+-----------------------+
|Danielle  |Johnson  |University of Alabama |Soccer            |2026-05-28 16:49:07.908|
|Jeffrey   |Lawrence |University of Georgia |Football          |2026-05-28 16:49:07.908|
|Teresa    |Gray     |University of Alabama |Soccer            |2026-05-28 16:49:07.908|
|Cynthia   |Diaz     |University of Michigan|Women's Basketball|2026-05-28 16:49:07.908|
|Richard   |Jones    |University of Alabama |Baseball          |2026-05-28 16:49:07.908|
+----------+---------+----------------------+------------------+-----------------------+

